# Agentic AI: Research (Tool use and Reflective agents)

- chain steps into research pipeline (search -> reflection -> formatting)
- convert natural language output into styled HTML

### Import libraries and load environment

In [1]:
import json
import requests

from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, HTML

import research_tools as rtools

load_dotenv()
client = OpenAI()

### Test arXiv search tool

In [2]:
results = rtools.arxiv_search("retrieval-augmented generation", max_results=3)

for i, paper in enumerate(results, 1):
  if 'error' in paper:
    display(HTML(f"<b>Error retrieving paper {i}:</b> {paper['error']}"))
    continue
  else:
    print(f"📄 Paper {i}")
    print(f"  Title     : {paper['title']}")
    print(f"  Authors   : {', '.join(paper['authors'])}")
    print(f"  Published : {paper['published']}")
    print(f"  URL pdf   : {paper['url_pdf']}\n")

print("\n🧾 Raw Results:\n")
print(json.dumps(results, indent=2))

📄 Paper 1
  Title     : AR-RAG: Autoregressive Retrieval Augmentation for Image Generation
  Authors   : Jingyuan Qi, Zhiyang Xu, Qifan Wang, Lifu Huang
  Published : 2025-06-08
  URL pdf   : https://arxiv.org/pdf/2506.06962v3

📄 Paper 2
  Title     : EVOR: Evolving Retrieval for Code Generation
  Authors   : Hongjin Su, Shuyang Jiang, Yuhang Lai, Haoyuan Wu, Boao Shi, Che Liu, Qian Liu, Tao Yu
  Published : 2024-02-19
  URL pdf   : https://arxiv.org/pdf/2402.12317v2

📄 Paper 3
  Title     : Automated Literature Review Using NLP Techniques and LLM-Based Retrieval-Augmented Generation
  Authors   : Nurshat Fateh Ali, Md. Mahdi Mohtasim, Shakil Mosharrof, T. Gopi Krishna
  Published : 2024-11-27
  URL pdf   : https://arxiv.org/pdf/2411.18583v1


🧾 Raw Results:

[
  {
    "title": "AR-RAG: Autoregressive Retrieval Augmentation for Image Generation",
    "authors": [
      "Jingyuan Qi",
      "Zhiyang Xu",
      "Qifan Wang",
      "Lifu Huang"
    ],
    "published": "2025-06-08",
    "u

### Test Tavilly search tool

In [3]:
# Test the Tavily search tool
topic = "retrieval-augmented generation applications"

tavily_results = rtools.tavily_search(topic)
for item in tavily_results:
    print(item)

{'title': 'What is Retrieval Augmented Generation (RAG)? - Databricks', 'content': '* Learn how retrieval augmented generation (RAG) works by combining large language models (LLMs) with real-time, external data for more accurate and relevant outputs. Retrieval augmented generation (RAG) is a hybrid AI framework that bolsters large language models (LLMs) by combining them with external, up-to-date data sources. This process flow helps developers update data sources without retraining the model and makes RAG a scalable and cost-effective solution for building LLM applications in domains like customer support, knowledge bases and internal search. With RAG architecture, organizations can deploy any LLM model and augment it to return relevant results for their organization by giving it a small amount of their data without the costs and time of fine-tuning or pretraining the model.', 'url': 'https://www.databricks.com/glossary/retrieval-augmented-generation-rag'}
{'title': 'Top Use Cases of 

### Tool mapping  

In [4]:
TOOL_MAPPING = {
  "arxiv_search": rtools.arxiv_search,
  "tavily_search": rtools.tavily_search
}

### functions

In [5]:
def generate_research_report(prompt: str, model: str = "gpt-4o") -> str:
    """
    Generates a research report using OpenAI's tool-calling with arXiv and Tavily tools.

    Args:
        prompt (str): The user prompt.
        model (str): OpenAI model name.

    Returns:
        str: Final assistant research report text.
    """

    messages = [
        {
            "role": "system", 
            "content": (
                "You are a research assistant that can search the web and arXiv to write detailed, "
                "accurate, and properly sourced research reports.\n\n"
                "🔍 Use tools when appropriate (e.g., to find scientific papers or web content).\n"
                "📚 Cite sources whenever relevant. Do NOT omit citations for brevity.\n"
                "🌐 When possible, include full URLs (arXiv links, web sources, etc.).\n"
                "✍️ Use an academic tone, organize output into clearly labeled sections, and include "
                "inline citations or footnotes as needed.\n"
                "🚫 Do not include placeholder text such as '(citation needed)' or '(citations omitted)'."
            )
        },
        {"role": "user", "content": prompt}
    ]

    tools = [rtools.arxiv_tool_def, rtools.tavily_tool_def]
    max_turns = 10

    for _ in range(max_turns):
        response = client.chat.completions.create( 
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            temperature=1, 
        ) 

        message = response.choices[0].message
        messages.append(message)

        if not message.tool_calls:      
            final_text = message.content
            print("✅ Final answer:")
            print(final_text)
            break

        if message.tool_calls is None:
            break

        for call in message.tool_calls:
            tool_name = call.function.name
            tool_args = json.loads(call.function.arguments)
            print(f"🔧 Calling tool: {tool_name} with args {tool_args}")

            try:
                tool_func = TOOL_MAPPING[tool_name]
                result = tool_func(**tool_args)
            except Exception as e:
                result = {"error": str(e)}
            
            new_msg = {
                "role": "tool",
                "tool_call_id": call.id,
                "name": tool_name,
                "content": json.dumps(result)
            }
            messages.append(new_msg)

    return final_text

In [6]:
def reflection_rewrite(report, model: str = "gpt-4o-mini", temperature: float = 0.3) -> dict:
    """
    Generates a structured reflection AND a revised research report.
    Accepts raw text OR the messages list returned by generate_research_report_with_tools.

    Returns:
        dict with keys:
          - "reflection": structured reflection text
          - "revised_report": improved version of the input report
    """

    # can be plain text or array of messages, parse_input handles both
    report = rtools.parse_input(report)

    prompt = f"""
    You are an academic reviewer and editor.
    Retrun only valid json with exacctly these keys:
    [{{"reflection": "<text>", "revised_report": "<text>"}}]
    No markkdown fences, no extra commentary.

    Guidelines:
    - Reflection must include: Strengths, Limitations, Suggestions, Opportunities.
    - Revised report should incorporate the suggestions and improve clarity and academic tone.

    Report:
    {report}
    """

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are an academic reviewer and editor."},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature,
    )

    result = response.choices[0].message.content.strip()
    
    try:
        data = json.loads(result)
        if isinstance(data, list):
            if len(data) == 0:
                raise ValueError("Model returned an empty list.")
            data = data[0]  # unwrap first element

    except json.JSONDecodeError:
        raise Exception("The output of the LLM was not valid JSON. Adjust your prompt.")

    return {
        "reflection": str(data.get("reflection", "")).strip(),
        "revised_report": str(data.get("revised_report", "")).strip(),
    }

### Format the output

In [7]:
def convert_to_html(report, model: str = "gpt-4o", temperature: float = 0.5) -> str:
    """
    Converts a research report into HTML format using an LLM.

    Args:
        report (str): The research report text.
        model (str): OpenAI model name.
        temperature (float): Sampling temperature.

    Returns:
        str: HTML formatted report.
    """
    report = rtools.parse_input(report)

    prompt = f"""
    You are an expert technical writing assistant.
    Convert the following plaintext research report into a clean, structured HTML document.
    Include section headers, well-formatted paragraphs, and clickable links.
    Ensure citation style is preserved.

    Respond ONLY with valid HTML (no explanation).

    Report:
    {report}
    """

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You convert plaintext reports into full clean HTML documents."},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature,
    )

    html_output = response.choices[0].message.content.strip()
    return html_output

### Run end to end

In [8]:
# 1 Research topic
research_topic = "applications of retrieval-augmented generation in healthcare"
preliminary_report = generate_research_report(research_topic)

print("\n\n--- Preliminary Report ---\n")
print(preliminary_report)

# 2 Reflection and rewrite
reflection_results = reflection_rewrite(preliminary_report)
print("\n\n--- Reflection ---\n")
print(reflection_results["reflection"])
print("\n\n--- Revised Report ---\n")
print(reflection_results["revised_report"])

# 3 Convert to HTML
html_report = convert_to_html(reflection_results["revised_report"])
print("=== Generated HTML (preview) ===\n")
print((html_report or "")[:600], "\n... [truncated]\n")

# 4) Display full HTML
display(HTML(html_report))


🔧 Calling tool: arxiv_search with args {'query': 'retrieval-augmented generation healthcare', 'max_results': 5}
🔧 Calling tool: tavily_search with args {'query': 'applications of retrieval-augmented generation in healthcare', 'max_results': 5}
✅ Final answer:
## Applications of Retrieval-Augmented Generation in Healthcare

Retrieval-Augmented Generation (RAG) is a technique that integrates external data sources with generative models to produce more accurate and contextually relevant outputs. This approach has numerous applications in the healthcare domain, leveraging the vast amounts of medical data to enhance clinical decision-making and research.

### Overview of RAG in Healthcare

1. **Clinical Decision Support**: RAG systems can assist healthcare professionals by providing data-driven support for clinical decisions. By retrieving relevant medical literature and integrating it into the decision-making process, RAG can help clinicians interpret guidelines more accurately and apply e